# Recurrences and Recursive Cost


```{contents}
:local:
:depth: 2
```

```{index} Account for the Children and the Work Around Them
```



```{index} Account for the Children and the Work Around Them
```

## Account for the Children and the Work Around Them



A recurrence models the work of a recursive algorithm. It is not the algorithm's return-value equation unless the counted work happens to satisfy the same relation. A recursive sum may return a monetary total while its call count follows a completely different equation. Name the quantity being modeled and include the base cost.

For a one-step reduction, suppose C(0)=1 and C(n)=1+C(n-1). Unfolding gives C(n)=1+1+C(n-2), then n non-base contributions plus the base contribution. Thus C(n)=n+1 under this counting convention. Omitting the base call changes exact counts but not linear growth; state the convention so measured results and formulas can agree.

A halving recursion has a different depth. If each call makes one child of size floor(n/2), the active chain has logarithmic length, but a call that performs linear local work still contributes that work at every level. Do not confuse a halving input with automatically logarithmic total time.

| Recursive design | Work at each level | Number of levels | Resulting pattern |
| --- | --- | --- | --- |
| One child smaller by one, constant local work | Constant | Linear | Linear total |
| One half-size child, constant local work | Constant | Logarithmic | Logarithmic total |
| Two half-size children, linear combine | About n across a level | Logarithmic | n log n total |

For related explanation, see the further reading below.[^development-reading]


```{index} Track Rounding and Live Storage
```



```{index} Track Rounding and Live Storage
```

## Track Rounding and Live Storage



For odd n, two subproblems cannot both have size n/2 as an integer. A range split may have floor(n/2) and ceil(n/2) items. Together they must still cover the whole input without overlap or omission. Bounds can simplify rounding when justified, but an implementation must handle exact endpoints.

In a balanced two-child recursion, a level can contain many calls even though only one root-to-leaf path is active at a time in depth-first execution. Total calls, maximum active frames and allocated buffers are separate measurements. Merge sort's reusable buffer and recursive frames contribute different terms to auxiliary space. Cumulative writes also differ from simultaneous storage.

Overlapping subproblems change the count again. A naive Fibonacci recursion revisits the same numeric arguments through multiple branches. Memoization can collapse repeated computation into distinct solved states, but the first recursive path still needs stack space. Chapter 25 develops that transformation; a recurrence analysis should identify the repeated work rather than claiming that every branching recursion has the same cost.

Guided exercise: list the subproblem sizes for a balanced split of five items, continuing to singletons. Check that each level's item coverage is correct, count calls under an explicit base convention, and distinguish the deepest chain from the entire tree of calls. Then compare this tree with one that creates only a single half-size child per call.


A recurrence models work by input size. Keep the returned value, total calls, and maximum simultaneously active frames separate. These examples preview recursive analysis; Chapter 22 develops recursive programming.

```{index} Summations You Will Meet
```

### Summations You Will Meet

Loop counts and recurrence solutions keep producing the same few sums, so it pays to know their closed forms and to recognize the shape of each.

| Sum | Closed form | Where it comes from |
| --- | --- | --- |
| 1 + 2 + ... + n | n(n + 1) / 2 | A triangular loop; insertion sort's worst case |
| 1 + 2 + 4 + ... + 2ᵏ | 2ᵏ⁺¹ - 1 | Nodes in a perfect binary tree; copies made by doubling growth |
| 1 + 1/2 + 1/4 + ... | approaches 2 | Bottom-up heap construction; any series whose terms halve |
| n + n + ... + n, log₂ n times | n log₂ n | One level of merge sort times its number of levels |
| 1² + 2² + ... + n² | n(n + 1)(2n + 1) / 6 | Three nested dependent loops |

The first two are the arithmetic and geometric series. In the arithmetic series the terms grow by a constant difference and the total is about half the square of the last term. In a geometric series with ratio 2 the terms double, and the total is dominated by the last term: the whole sum is less than twice it. That one fact explains why doubling an array costs only O(n) in total copies and why bottom-up heap construction is linear. The example verifies each formula against a direct loop.

In [ ]:
using System;

foreach (int n in new[] { 1, 7, 8, 100 })
{
    long tri = 0, squares = 0;
    for (int i = 1; i <= n; i++) { tri += i; squares += (long)i * i; }
    int k = (int)Math.Floor(Math.Log2(n));
    long geo = 0;
    for (int i = 0; i <= k; i++) geo += 1L << i;
    Console.WriteLine($"n={n,3}: 1+..+n={tri,5} formula={(long)n * (n + 1) / 2,5} | 1+2+..+2^{k}={geo,4} formula={(1L << (k + 1)) - 1,4} | squares={squares,7} formula={(long)n * (n + 1) * (2 * n + 1) / 6,7}");
}

n=  1: 1+..+n=    1 formula=    1 | 1+2+..+2^0=   1 formula=   1 | squares=      1 formula=      1
n=  7: 1+..+n=   28 formula=   28 | 1+2+..+2^2=   7 formula=   7 | squares=    140 formula=    140
n=  8: 1+..+n=   36 formula=   36 | 1+2+..+2^3=  15 formula=  15 | squares=    204 formula=    204
n=100: 1+..+n= 5050 formula= 5050 | 1+2+..+2^6= 127 formula= 127 | squares= 338350 formula= 338350


Every row matches its formula. Notice how the geometric sum relates to its last term: for n = 100 the last power is 64 and the sum is 127, just under twice 64, whatever the length of the series. The arithmetic sum, by contrast, is about n / 2 times its last term. When a count is a sum of terms that shrink geometrically, the total is a constant multiple of the largest term; when the terms grow linearly, the total is quadratic. Recognizing which situation a loop produces is most of recurrence solving.

```{index} Solve a Recurrence by Expanding
```

### Solve a Recurrence by Expanding

A recurrence defines a cost in terms of the same cost on a smaller input, and the direct way to solve one is to *expand* it: substitute the recurrence into itself until a pattern appears, then sum. Take the cost of a method that does n units of work and then calls itself on n - 1, with T(0) = 0.

```text
T(n) = n + T(n - 1)
     = n + (n - 1) + T(n - 2)
     = n + (n - 1) + (n - 2) + ... + 1 + T(0)
     = n(n + 1) / 2
```

Now the halving recurrence with linear work at each level, T(1) = 1 and T(n) = 2T(n / 2) + n for n a power of two:

```text
T(n) = n + 2T(n / 2)
     = n + 2(n / 2 + 2T(n / 4)) = n + n + 4T(n / 4)
     = n + n + n + 8T(n / 8)
     = n × log₂ n + n × T(1) = n log₂ n + n
```

Each expansion adds one more copy of n and doubles the number of subproblems, and the process stops after log₂ n steps when the subproblems reach size 1. The guessed closed form is then confirmed by induction, as 21.2 described: check T(1), assume the formula for n / 2, and show it yields the formula for n. The example expands both recurrences mechanically and compares the result with the closed forms at several sizes.

In [ ]:
using System;

Console.WriteLine("  n   T(n)=n+T(n-1)  n(n+1)/2   T(n)=2T(n/2)+n  n*log2(n)+n");
foreach (int n in new[] { 1, 2, 8, 64, 1024 })
    Console.WriteLine($"{n,5} {Linear(n),14} {(long)n * (n + 1) / 2,10} {Halving(n),16} {(long)(n * Math.Log2(n)) + n,13}");

static long Linear(int n) => n == 0 ? 0 : n + Linear(n - 1);
static long Halving(int n) => n == 1 ? 1 : 2 * Halving(n / 2) + n;

  n   T(n)=n+T(n-1)  n(n+1)/2   T(n)=2T(n/2)+n  n*log2(n)+n
    1              1          1                1             1
    2              3          3                4             4
    8             36         36               32            32
   64           2080       2080              448           448
 1024         524800     524800            11264         11264


Both columns agree with their closed forms at every size, which is what the expansion predicted and the induction would confirm. The two recurrences differ in one place, whether the smaller problem is n - 1 or n / 2, and that single difference separates a quadratic cost from an n log n one: at n = 1,024 the first is 524,800 and the second 11,264. This is the arithmetic behind the table at the top of this page, and the reason divide-and-conquer earns its name.

```{index} Concept Check: Match the Recurrence to Its Growth
```

### Concept Check: Match the Recurrence to Its Growth

Match each recurrence, with T(1) = 1, to its growth class, then open the answer.

1. T(n) = T(n - 1) + 1
2. T(n) = T(n / 2) + 1
3. T(n) = T(n - 1) + n
4. T(n) = 2T(n / 2) + n
5. T(n) = 2T(n / 2) + 1
6. T(n) = 2T(n - 1) + 1

```{dropdown} Answer
1. Θ(n): one unit per level, n levels (a countdown or a linear scan).
2. Θ(log n): one unit per level, log n levels (binary search).
3. Θ(n²): n units per level, n levels (the expansion above; selection sort).
4. Θ(n log n): n units per level, log n levels (merge sort).
5. Θ(n): the work is in the leaves; 2ᵏ subproblems after k levels gives n leaves of unit cost (counting the nodes of a tree).
6. Θ(2ⁿ): each level doubles the number of calls and there are n levels (the Towers of Hanoi; naive branching recursion).

Recurrences 4 and 5 differ only in the combine work and land one class apart; 1 and 6 differ only in the branching and land many classes apart.
```

```{index} Code Tracing: How Many Calls?
```

### Code Tracing: How Many Calls?

For each method, write a recurrence for the number of calls C(n), including the base call, and predict C(6). Then run the solution, which counts the calls directly.

```csharp
static int A(int n) => n <= 0 ? 0 : 1 + A(n - 2);
static int B(int n) => n <= 1 ? n : B(n - 1) + B(n - 2);
static int C(int n) => n <= 1 ? 1 : C(n / 2) + C(n / 2);
static int D(int n) => n <= 1 ? 1 : D(n / 2) + D(n / 3);
```

In [ ]:
// Solution
using System;

int calls = 0;
foreach (int n in new[] { 6, 20 })
{
    calls = 0; A(n); long a = calls;
    calls = 0; B(n); long b = calls;
    calls = 0; C(n); long c = calls;
    calls = 0; D(n); long d = calls;
    Console.WriteLine($"n={n,2}: A calls={a,5}  B calls={b,6}  C calls={c,5}  D calls={d,5}");
}

int A(int n) { calls++; return n <= 0 ? 0 : 1 + A(n - 2); }
int B(int n) { calls++; return n <= 1 ? n : B(n - 1) + B(n - 2); }
int C(int n) { calls++; return n <= 1 ? 1 : C(n / 2) + C(n / 2); }
int D(int n) { calls++; return n <= 1 ? 1 : D(n / 2) + D(n / 3); }

n= 6: A calls=    4  B calls=    25  C calls=    7  D calls=    7
n=20: A calls=   11  B calls= 21891  C calls=   31  D calls=   17


A satisfies C(n) = 1 + C(n - 2) with C(0) = 1, so C(6) = 4 and the growth is Θ(n) with half the constant of a countdown. B is the naive Fibonacci recurrence, C(n) = 1 + C(n - 1) + C(n - 2), 25 calls for n = 6 and 21,891 for n = 20: exponential. C makes two half-size calls with no other work, C(n) = 1 + 2C(n / 2), so its calls roughly double with n, 2n - 1 for powers of two, Θ(n); the second call repeats the first and a memo of one result would halve the work. D splits into halves and thirds, C(n) = 1 + C(n / 2) + C(n / 3); because 1 / 2 + 1 / 3 is less than 1 the total work is dominated by the top levels and grows more slowly than linearly, which the counts for 6 and 20 begin to show. Comparing C and D makes the general principle visible: what matters is whether the subproblem sizes add up to the original, to more, or to less.

```{index} Base Cases and Linear Recurrences
```

## Base Cases and Linear Recurrences




```{index} Base Cases and Linear Recurrences
```


Let C(n) count calls, including the base call. For a nonnegative countdown, C(0)=1 and C(n)=C(n−1)+1. Expanding gives C(n)=n+1. The returned value below is n, which is a different quantity.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0,1,4 })
{
    int calls=0, active=0, peak=0;
    int result=Count(n);
    Console.WriteLine($"n={n}, result={result}, calls={calls}, peak frames={peak}");
    int Count(int size)
    {
        calls++; active++; peak=Math.Max(peak,active);
        int value = size == 0 ? 0 : 1 + Count(size-1);
        active--; return value;
    }
}

n=0, result=0, calls=1, peak frames=1
n=1, result=1, calls=2, peak frames=2
n=4, result=4, calls=5, peak frames=5


Time and stack space are Θ(n+1); for growing positive n, write Θ(n). Each call has constant local storage. Negative input would never reach zero in this model, so nonnegative input is a required precondition. Production recursive code needs a practical depth limit; .NET does not promise unlimited recursion or tail-call elimination.

**Common mistake.** Writing the recurrence without its base case, `C(n) = C(n - 1) + 1` alone. It has no solution, since nothing anchors `C(0)`; the base call is one unit of work and the closed form `n + 1` depends on it.


```{index} Halving with Integer Inputs
```

## Halving with Integer Inputs




```{index} Halving with Integer Inputs
```


C(n)=1 for n≤1 and C(n)=C(floor(n/2))+1 otherwise. For n≥1 this gives floor(log₂ n)+1 calls. The number of reductions is one less than the number of calls.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0,1,3,8,9 })
    Console.WriteLine($"n={n}, calls={HalvingCalls(n)}");
int HalvingCalls(int size)
{
    if (size < 0) throw new ArgumentOutOfRangeException(nameof(size));
    return size <= 1 ? 1 : 1 + HalvingCalls(size/2);
}

n=0, calls=1
n=1, calls=1
n=3, calls=2
n=8, calls=4
n=9, calls=4


For powers of two, the chain is n, n/2, …, 1. For n=9 the integer chain is 9,4,2,1. Depth and time are Θ(log n) for large n; zero and one remain constant-cost base cases.

**Common mistake.** Assuming the chain for `n = 9` is 9, 4.5, 2.25, ... and reporting a fractional depth. Integer division gives 9, 4, 2, 1, so the call count is `floor(log₂ 9) + 1 = 4`; the model must use the same arithmetic as the code.


```{index} Two Branches and Work per Level
```

## Two Branches and Work per Level




```{index} Two Branches and Work per Level
```


For a model with n=2^k leaves, define W(1)=1 and W(n)=2W(n/2)+n. At every internal level, the total combine work is n. There are log₂ n internal levels, plus n leaf units, so W(n)=n log₂ n+n. This models a merge-style algorithm without implementing sorting here.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 1,2,4,8,16 })
{
    long calls=0; int active=0, peak=0;
    long work=Work(n);
    Console.WriteLine($"n={n}, work={work}, calls={calls}, peak frames={peak}");
    long Work(int size)
    {
        calls++; active++; peak=Math.Max(peak,active);
        long result = size == 1 ? 1 : Work(size/2)+Work(size/2)+size;
        active--;return result;
    }
}

n=1, work=1, calls=1, peak frames=1
n=2, work=4, calls=3, peak frames=2
n=4, work=12, calls=7, peak frames=3
n=8, work=32, calls=15, peak frames=4
n=16, work=80, calls=31, peak frames=5


For n=8, level work is 8+8+8+8=32, total calls are 15, and peak active frames are 4. Sibling calls occur sequentially, so they increase total calls without doubling stack depth. This counter program itself visits Θ(n) nodes; its returned work models an algorithm that actually performs n combine operations at each internal node.

**Common mistake.** Adding the two child costs and forgetting the work around them, `W(n) = 2W(n/2)`. That recurrence solves to Θ(n), the leaf count, and misses the `n` units of combining at every level that make the total `n log₂ n + n`.


```{index} Rounding and Auxiliary Storage
```

## Rounding and Auxiliary Storage




```{index} Rounding and Auxiliary Storage
```


For arbitrary positive sizes, split into floor(n/2) and ceil(n/2), preserving all items. Using floor(n/2) twice loses an item for odd n. For n>1, both parts must be smaller than n.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 1,3,5,8,9 })
    Console.WriteLine($"n={n}, modeled work={BalancedWork(n)}");
long BalancedWork(int n)
{
    if(n < 1) throw new ArgumentOutOfRangeException(nameof(n));
    if(n == 1) return 1;
    int left=n/2, right=n-left;
    return BalancedWork(left)+BalancedWork(right)+n;
}

n=1, modeled work=1
n=3, modeled work=8
n=5, modeled work=17
n=8, modeled work=32
n=9, modeled work=38


The general model remains Θ(n log n) work. Its stack depth is Θ(log n), with constant storage per frame. A real merge sort usually needs Θ(n) auxiliary merge storage as well. Stack depth alone is not total auxiliary space. State how buffers are allocated and reused before claiming a memory bound.

**Common mistake.** Splitting an odd length as `n/2` and `n/2`. For `n = 5` the two halves hold four items; one item is dropped on every odd split, and the recursion still terminates, so nothing signals the loss except a preservation check.


```{index} Branching and Repeated Subproblems
```

## Branching and Repeated Subproblems




```{index} Branching and Repeated Subproblems
```


The naive Fibonacci call count obeys C(0)=C(1)=1 and C(n)=C(n−1)+C(n−2)+1. Repeated subproblems create exponential work. A recursion tree can be wide even when its depth is linear.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0,1,4,6,8 })
{
    long calls=0;int active=0,peak=0;
    int value=Fib(n);
    Console.WriteLine($"n={n}, Fib={value}, calls={calls}, peak frames={peak}");
    int Fib(int size)
    {
        calls++;active++;peak=Math.Max(peak,active);
        int result=size<=1?size:Fib(size-1)+Fib(size-2);
        active--;return result;
    }
}

n=0, Fib=0, calls=1, peak frames=1
n=1, Fib=1, calls=1, peak frames=1
n=4, Fib=3, calls=9, peak frames=4
n=6, Fib=8, calls=25, peak frames=6
n=8, Fib=21, calls=67, peak frames=8


For n=6 there are 25 calls but only 6 active frames at the peak. The time is O(2^n), a useful upper bound; the tighter exponential growth is Θ(φ^n), where φ≈1.618. Stack space is Θ(n). Memoization reduces repeated work by retaining results; its implementation belongs in Chapter 25. Derive a recurrence from calls and nonrecursive work, include its base case, and distinguish the model from the code that calculates an estimate.

**Common mistake.** Reading the 25 calls for `n = 6` as 25 simultaneous frames. The peak is 6, because sibling calls run one after the other; call count measures time and peak frames measure stack space, and the two grow at very different rates here.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. Why does a geometric series with ratio 1/2 stay bounded by a constant multiple of its first term, and which algorithm's cost argument depends on that?

```{dropdown} Answer
Each term is half the previous one, so all the terms after the first add up to less than the first term again, and the whole series is under twice the first term. Bottom-up heap construction depends on it: the nodes that need k levels of sinking are halved at each k, so the total work is linear in n.
```

2. Two recurrences, T(n) = 2T(n / 2) + n and T(n) = 2T(n / 2) + n², look alike. Why are their solutions so different?

```{dropdown} Answer
In the first, every level does the same n units of work, so the total is n times the number of levels. In the second, the top level does n² and the next level only n² / 2 in total, halving at each level; the series is geometric and the root dominates, giving Θ(n²). Where the work concentrates, at the leaves, evenly, or at the root, decides the class.
```

3. A recursive method's call count is 2ⁿ but its peak stack depth is n. Which one limits the input size in practice, and why?

```{dropdown} Answer
Time usually does. A depth of a few thousand frames is fine, but 2ⁿ calls exceed any budget long before n reaches 50. The depth would matter first only if the method were linear in time and the input very large, as in a recursion over a long list.
```

4. You have guessed that T(n) = n log₂ n + n solves T(n) = 2T(n / 2) + n. What are the two steps that turn the guess into a proof?

```{dropdown} Answer
Check the base case, T(1) = 1 log₂ 1 + 1 = 1. Then assume the formula for n / 2, substitute it into the recurrence, 2((n / 2) log₂(n / 2) + n / 2) + n, and simplify to n log₂ n + n. That is induction on n, the same argument shape as a loop invariant.
```

```{index} Exercise
```

## Exercise




```{index} Analysis practice
```

For n=8, print the number of leaves and total calls in a balanced two-branch recursion with constant work per node. Contrast this with a linear combine-cost model.

In [ ]:
using System;

// Exercise
// For n=8, print the number of leaves and total calls in a balanced two-branch recursion with constant work per node. Contrast this with a linear combine-cost model.
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;

// Solution
int calls=0;
Console.WriteLine($"leaves={Leaves(8)}, calls={calls}");
int Leaves(int n)
{
 calls++;
 return n==1?1:Leaves(n/2)+Leaves(n/2);
}
// C(1)=1, C(n)=2C(n/2)+1; Theta(n) calls, Theta(log n) stack.
// A linear combine cost changes modeled work to Theta(n log n).

leaves=8, calls=15


The output is `leaves=8, calls=15`: for `n = 8` there are `n` leaves and `2n - 1` calls in total, while the deepest stack holds only four frames. Add a linear combine step at each internal node and the modeled work becomes `n log₂ n + n`, as the Two Branches section computed.


```{rubric} Footnotes
```


[^2103_recurrences-1]: Recursion depth here counts active calls including the base call. Some texts count edges between calls instead; specify the convention.
[^2103_recurrences-2]: The power-of-two assumption simplifies exact level sums. Balanced integer splits retain the same asymptotic bound for other positive sizes.

[^development-reading]: Further reading: [OpenDSA CS3, Common Misunderstandings](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/AnalMisunderstanding.html). The worked business scenarios and guided reasoning here are original teaching examples.
